# Setup Agentic Information Retrieval (IR) Multi-Agent System
Questo notebook è configurato per l'esecuzione su **Google Colab** (consigliato: acceleratore **T4 GPU** in Runtime -> Cambia tipo di runtime).

Il sistema implementa un'architettura multi-agente ciclica basata su:
- **Adaptive-RAG** (Jeong et al., 2024): classificazione della complessità e routing dinamico tra Knowledge Base locale di Wikipedia e Web Search.
- **Self-RAG** (Asai et al., 2023): critique tokens (`is_context_relevant`, `is_query_answered`) per garantire il grounding ed evitare allucinazioni.
- **Reflexion** (Shinn et al., 2023): memoria episodica verbale per l'auto-correzione tra iterazioni successive.
- **Least-to-Most & Query2doc** (Zhou et al., 2022; Wang et al., 2023): scomposizione ed espansione semantica delle query nel Refiner.
- **ReAct & IRCoT** (Yao et al., 2022; Trivedi et al., 2022): recupero incrementale e ragionamento interleaved nel Planner.

In [ ]:
# 1. Verifica disponibilità GPU NVIDIA
!nvidia-smi

# 2. Installazione runtime CUDA 12 per linking dinamico (risolve libcudart.so.12)
!pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12

# 3. Installazione librerie core di orchestrazione e ricerca
!pip install -q langchain langgraph langchain-community huggingface-hub pydantic datasets rank_bm25
!pip install -q -U duckduckgo-search beautifulsoup4 requests pandas

# 4. Installazione di llama-cpp-python precompilato con supporto CUDA 12 per GPU Colab
wheel_url = "https://abetlen.github.io/llama-cpp-python/whl/cu124"
!pip install llama-cpp-python --extra-index-url {wheel_url} -q

## Clonazione e Sincronizzazione della Repository
Scarichiamo o sincronizziamo il codice sorgente aggiornato della repository GitHub.

In [ ]:
import os

REPO_URL = "https://github.com/GabCovone/AgenticAskMe"
repo_name = REPO_URL.split("/")[-1].replace(".git", "")

if not os.path.exists(repo_name):
    !git clone {REPO_URL}
    os.chdir(repo_name)
else:
    os.chdir(repo_name)
    !git pull

print(f"Directory di lavoro attiva: {os.getcwd()}")

## Inizializzazione della Knowledge Base Locale e del Grafo Multi-Agente
1. Inizializziamo il database SQLite locale (`wikipedia_local.db`) con indice full-text (FTS5) e BM25Okapi in cache.
2. Inizializziamo il modello LLM (`Mistral-Nemo-Instruct` o `Gemma-2-9B-it`) con **temperatura a 0.3** per garantire flessibilità generativa e precisione nello schema JSON.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', message='.*utcnow.*')
warnings.filterwarnings('ignore', module='.*jupyter_client.*')

import local_kb
from graph import build_graph

# 1. Inizializzazione immediata della Knowledge Base locale (SQLite FTS5 + BM25Okapi)
local_kb.init_db(force_reseed=False)
kb_info = local_kb.get_stats()
print(f"Knowledge Base locale attiva: {kb_info['total_articles']} articoli indicizzati in {kb_info['db_path']}.")

# Opzionale: per scaricare un sottoinsieme del 5% dell'intera Wikipedia da HuggingFace (richiede più tempo e RAM):
# local_kb.build_index(subset_percent=5)

# 2. Compilazione del Grafo LangGraph (modello con temperature=0.3 e dual-retrieval Local KB / Web)
# build_graph() precarica automaticamente le librerie CUDA di runtime (libcudart.so.12 / libcublas.so.12)
app = build_graph()
print("Grafo multi-agente compilato e pronto all'uso!")

## Test Interattivo su Query Singola
Testiamo una domanda per osservare l'intera traiettoria agentica:
- **Validator**: valutazione della complessità e routing (`route_to_refinement` o `route_to_planning`)
- **Refiner**: scomposizione Least-to-Most o espansione Query2doc
- **Planner**: Adaptive-RAG (scelta tra Local Wikipedia KB e Web Search) + ReAct multi-step
- **Validator**: controllo di aderenza Self-RAG (IsRel, IsSup) e verdetto finale

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', message='.*utcnow.*')
warnings.filterwarnings('ignore', module='.*jupyter_client.*')

# Inserisci qui una domanda di test:
test_query = "Were Scott Derrickson and Ed Wood of the same nationality?"

initial_state = {
    "original_query": test_query,
    "current_query": test_query,
    "retrieved_context": "",
    "num_refinement": 0,
    "num_planning": 0,
    "next_node": "",
    "feedback_history": [],
    "final_answer": ""
}

print(f"=== AVVIO TEST INTERATTIVO PER: '{test_query}' ===\n")
final_output = ""

for output in app.stream(initial_state, {"recursion_limit": 30}):
    for node_name, state_update in output.items():
        print(f"\n>>> Passaggio per il nodo: [{node_name.upper()}]")
        if "next_node" in state_update:
            print(f"    Routing verso: {state_update['next_node'].upper()}")
        if "current_query" in state_update and state_update["current_query"] != test_query:
            print(f"    Query raffinata: {state_update['current_query']}")
        if "final_answer" in state_update and state_update["final_answer"]:
            final_output = state_update["final_answer"]
            print(f"    Proposta risposta: {final_output}")

print("\n" + "="*60)
print(f"RISPOSTA FINALE SINTETIZZATA:\n{final_output}")
print("="*60)

## Benchmark Quantitativo su HotpotQA (Multi-Hop Evaluation)
Eseguiamo la valutazione automatica su un campione di domande multi-hop dal dataset gold-standard **HotpotQA** (`fullwiki`).

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', message='.*utcnow.*')
warnings.filterwarnings('ignore', module='.*jupyter_client.*')

from datasets import load_dataset
import time

print("Scaricamento del subset di validazione di HotpotQA...")
dataset = load_dataset("hotpotqa/hotpot_qa", "fullwiki", split="validation")

NUM_TESTS = 10  # Numero di campioni da testare nel benchmark
correct = 0
results_log = []

print(f"\n--- AVVIO BENCHMARK SU {NUM_TESTS} DOMANDE HOTPOTQA --- ")
for i in range(NUM_TESTS):
    sample = dataset[i]
    question = sample["question"]
    gold_answer = sample["answer"]
    
    print(f"\n[Test {i+1}/{NUM_TESTS}]")
    print(f"Domanda: {question}")
    print(f"Gold Answer: {gold_answer}")
    
    state = {
        "original_query": question,
        "current_query": question,
        "retrieved_context": "",
        "num_refinement": 0,
        "num_planning": 0,
        "next_node": "",
        "feedback_history": [],
        "final_answer": ""
    }
    
    final_answer = ""
    try:
        for output in app.stream(state, {"recursion_limit": 30}):
            for key, value in output.items():
                if "final_answer" in value and value["final_answer"]:
                    final_answer = value["final_answer"]
    except Exception as e:
        print(f"Errore durante l'esecuzione: {e}")
        final_answer = "Error"
        
    print(f"Risposta Agente: {final_answer}")
    
    # Valutazione di aderenza semantica (Substring Match)
    is_correct = gold_answer.lower() in final_answer.lower()
    if is_correct:
        print("Valutazione: CORRETTA [OK]")
        correct += 1
    else:
        print("Valutazione: ERRATA [MISMATCH]")
        
    results_log.append({
        "question": question,
        "gold": gold_answer,
        "predicted": final_answer,
        "correct": is_correct
    })
    
    time.sleep(1) # Rispetto dei limiti di frequenza per la ricerca web

accuracy = (correct / NUM_TESTS) * 100
print("\n" + "="*60)
print(f"RISULTATI FINALI BENCHMARK: {correct}/{NUM_TESTS} corrette.")
print(f"ACCURACY (Substring Match): {accuracy:.1f}%")
print("="*60 + "\n")